# បកប្រែដោយ NLLB-200 លើ Google Colab → គេហទំព័រ KhmerDub AI

Notebook នេះបើក **NLLB Translation API** នៅលើ Colab រួចបើក tunnel ឲ្យគេហទំព័រភ្ជាប់បាន។
វារត់នៅទីនេះ ព្រោះទំហំអង្គចងចាំរបស់ម៉ូឌែលធំជាងម៉ាស៊ីនបម្រើគេហទំព័រ (Render free មានតែ ៥១២ MB)។

**ម៉ូឌែល (mode)** — លំនាំដើមគឺម៉ូឌែលខ្លាំងជាងគេ ដើម្បីឲ្យការបកប្រែត្រូវល្អជាងគេ៖

| mode | ម៉ូឌែល | ពេលណាគួរប្រើ |
|---|---|---|
| **best** (លំនាំដើម) | `facebook/nllb-200-3.3B` | គុណភាពខ្ពស់ជាងគេ — ត្រូវការ GPU T4 |
| balanced | `facebook/nllb-200-distilled-1.3B` | ជិតស្មើ best តែស្រាលជាងពាក់កណ្តាល |
| fast | `facebook/nllb-200-distilled-600M` | លឿនជាងគេ តែគុណភាពទាបជាងគេ |

បើ GPU ផ្ទុកម៉ូឌែលដែលជ្រើសមិនបាន (អស់ memory ឬគ្មាន GPU) សេវាធ្លាក់ទៅម៉ូឌែលតូចជាងដោយស្វ័យប្រវត្តិ
ហើយរាយការណ៍នៅ cell ទី ៣ ថាកំពុងប្រើ mode ណា។ ការបើកលើកដំបូងយូរ ២–៦ នាទី (ទាញទម្ងន់)។

**របៀបប្រើ (រត់តាមលំដាប់)៖**

1. `Runtime → Change runtime type → T4 GPU` (សំខាន់ — បើគ្មាន GPU វាយឺតជាងច្រើន)
2. រត់ cell ទី ២ (ពិនិត្យ GPU)
3. រត់ cell ទី ៣ (បើក API + tunnel) — វាបង្ហាញ **URL** និង **API Key**
4. ចម្លង URL + API Key ទៅកាត «បកប្រែ · NLLB API» ក្នុងផ្ទាំងស្ទូឌីយោ រួចចុច «រក្សាទុក និងសាកល្បង»

⚠️ ទុក tab នេះចោលបើកចុះ។ បិទ tab ឬទុកចោល (idle) ប្រហែល ៩០ នាទី = session ដាច់ ហើយ
URL នោះស្លាប់ — បើក cell ទី ៣ ម្ដងទៀត រួច paste URL ថ្មីក្នុងកាត។

In [ ]:
# ជំហានទី ១ — ពិនិត្យថា GPU ត្រូវបានបើក
# បើឃើញ "គ្មាន GPU" សូមធ្វើ៖ Runtime → Change runtime type → T4 GPU
!nvidia-smi -L || echo "គ្មាន GPU — សូមបើក T4 GPU ក្នុង Runtime → Change runtime type"
import torch
print("GPU ប្រើបាន:", torch.cuda.is_available())

In [ ]:
# ជំហានទី ២ — បើក NLLB API + tunnel (បញ្ជាតែមួយ)
#
# API Key៖ ដាក់ key ផ្ទាល់ខ្លួននៅបន្ទាត់ខាងក្រោម (កែអក្សរនៅក្នុងសញ្ញាក្បៀស) ដើម្បីកុំឲ្យ
# វាប្តូររាល់ពេលបើក session ថ្មី — រួចអ្នកគ្រាន់តែប្តូរ **URL** ក្នុងកាតលើគេហទំព័រ។
import os
os.environ["NLLB_API_KEY"] = "khmerdub-nllb-key"   # ➜ ប្តូរជាអក្សរសម្ងាត់ផ្ទាល់ខ្លួនរបស់អ្នក
# ម៉ូឌែល៖ best = ខ្លាំងជាងគេ (លំនាំដើម) · balanced · fast
os.environ["NLLB_MODE"] = "best"

!curl -fsSL https://raw.githubusercontent.com/kitreaksa78-debug/Sal/main/tools/nllb-colab/colab-start.sh | sh

In [ ]:
# ជំហានទី ៣ — មើល URL + Key ម្តងទៀតពេលណាក៏បាន (API និង tunnel នៅដើរ background)
import json, os, urllib.request
print("URL  :", open("/content/nllb-api/tunnel-url.txt").read().strip() if os.path.exists("/content/nllb-api/tunnel-url.txt") else "(មិនទាន់មាន)")
print("Key  :", os.environ.get("NLLB_API_KEY"))
info = json.load(urllib.request.urlopen("http://127.0.0.1:8000/"))
print("Mode  :", info.get("mode"), "·", info.get("model"), "·", info.get("device"), "· loaded:", info.get("loaded"))
if info.get("fallback"):
    print("⚠️ ម៉ូឌែលខ្លាំងជាងគេផ្ទុកមិនបាន — កំពុងប្រើ mode តូចជាងជំនួស៖", info.get("mode"))
if info.get("loadError"):
    print("❌ ផ្ទុកម៉ូឌែលបរាជ័យ:", info.get("loadError"))

### បញ្ជីត្រួតពិនិត្យ (Checklist)

| ជំហាន | របៀបផ្ទៀងផ្ទាត់ |
|---|---|
| ១. GPU បើក | `torch.cuda.is_available()` → `True` |
| ២. API រត់ | cell ទី ៣ បង្ហាញ `device: cuda` និង `loaded: true` |
| ២.១ Mode | cell ទី ៣ បង្ហាញ `Mode : best · facebook/nllb-200-3.3B · cuda` |
| ៣. គេហទំព័រដឹង | កាត «បកប្រែ · NLLB API» បង្ហាញ **ភ្ជាប់រួច** |
| ៤. សាកល្បងជោគជ័យ | «រក្សាទុក និងសាកល្បង» បង្ហាញ `NLLB Translation API · cuda` |
| ៥. ការងារពិត | បកប្រែវីដេអូ → log ជំហានបកប្រែបង្ហាញ NLLB |

### បើមានបញ្ហា

| រោគសញ្ញា | ដំណោះស្រាយ |
|---|---|
| សាកល្បងលើគេហទំព័របរាជ័យ តែ cell នេះបង្ហាញ URL | URL ចាស់ (session មុនបានដាច់) — copy URL ថ្មីពី cell ទី ៣ |
| `device: cpu` | runtime គ្មាន GPU — បើក T4 រួចរត់ cell ទី ២ ម្ដងទៀត |
| `loaded: false` យូរពេក | ម៉ូឌែលកំពុងផ្ទុក (២–៦ នាទី ព្រោះទាញទម្ងន់ ៣.៣B) — រង់ចាំ ឬមើល `tail -f /content/nllb-api/api.log` |
| ចង់បានគុណភាពខ្ពស់ជាងនេះទៀត | ដាក់ `os.environ["NLLB_NUM_BEAMS"] = "4"` មុនរត់ cell ទី ២ (យឺតជាង ~៣ ដង) |
| ចង់ឲ្យរត់លឿន/ស្រាល | `os.environ["NLLB_MODE"] = "fast"` ឬ `"balanced"` |
| `401 Invalid or missing API key` | API Key ក្នុងកាតលើគេហទំព័រ មិនដូចនៅក្នុង cell ទី ២ |
